In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
import random
import os

# Define Image Size for InceptionV3
image_size = (299, 299)

# Load handcrafted features
feature_data = pd.read_csv("../Dataset/Features/signature_features.csv")
feature_data.iloc[:, 2:] = feature_data.iloc[:, 2:].apply(pd.to_numeric, errors='coerce')  # Convert all to numeric
feature_data.fillna(0, inplace=True)  # Fill NaN values with 0
feature_dict = {row["filename"]: row.iloc[2:].values.astype(np.float32) for _, row in feature_data.iterrows()}

# Define the Data Generator for Siamese Network
class SiameseSignatureDataGenerator(Sequence):
    def __init__(self, image_paths, labels, handcrafted_features, batch_size=32, shuffle=True):
        self.image_paths = image_paths
        self.labels = labels
        self.handcrafted_features = handcrafted_features
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.pairs = self.create_pairs()
        self.indexes = np.arange(len(self.pairs))
        self.on_epoch_end()

    def create_pairs(self):
        """Create pairs of (genuine, genuine) and (genuine, forged) for Siamese training."""
        pairs = []
        genuine_images = [p for p, l in zip(self.image_paths, self.labels) if l == 0]
        forged_images = [p for p, l in zip(self.image_paths, self.labels) if l == 1]

        for genuine in genuine_images:
            # Positive Pair (same class)
            positive_pair = random.choice(genuine_images)
            pairs.append((genuine, positive_pair, 0))

            # Negative Pair (genuine vs forged)
            negative_pair = random.choice(forged_images)
            pairs.append((genuine, negative_pair, 1))

        return pairs

    def __len__(self):
        return int(np.ceil(len(self.pairs) / self.batch_size))

    def __getitem__(self, index):
        batch_pairs = self.pairs[index * self.batch_size:(index + 1) * self.batch_size]
        
        images_1, images_2, handcrafted_1, handcrafted_2, labels = [], [], [], [], []

        for img1_path, img2_path, label in batch_pairs:
            img1 = self.load_image(img1_path)
            img2 = self.load_image(img2_path)

            feat1 = feature_dict.get(os.path.basename(img1_path), np.zeros(50))  # Default 50 feature size
            feat2 = feature_dict.get(os.path.basename(img2_path), np.zeros(50))

            images_1.append(img1)
            images_2.append(img2)
            handcrafted_1.append(feat1)
            handcrafted_2.append(feat2)
            labels.append(label)

        return ([np.array(images_1, dtype=np.float32), np.array(images_2, dtype=np.float32), 
            np.array(handcrafted_1, dtype=np.float32), np.array(handcrafted_2, dtype=np.float32)], 
            np.array(labels, dtype=np.float32))


    def load_image(self, image_path):
        img = load_img(image_path, target_size=image_size)
        img = img_to_array(img)
        return tf.keras.applications.inception_v3.preprocess_input(img)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# Create Dataset Paths
image_paths = [f"../Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{file}" 
               for file, label in zip(feature_data["filename"], feature_data["label"].values)]
labels = feature_data["label"].values

# Train-Test Split
train_img_paths, val_img_paths, train_labels, val_labels = train_test_split(image_paths, labels, test_size=0.2, random_state=42, stratify=labels)

# Create Data Generators
batch_size = 32
train_generator = SiameseSignatureDataGenerator(train_img_paths, train_labels, feature_dict, batch_size=batch_size)
val_generator = SiameseSignatureDataGenerator(val_img_paths, val_labels, feature_dict, batch_size=batch_size)


In [ ]:
from tensorflow.keras.layers import Input, Dense, LSTM, GlobalAveragePooling2D, Concatenate, Lambda
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
from tensorflow.keras.applications import InceptionV3

# CNN Backbone (InceptionV3)
def build_cnn_branch():
    base_model = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))
    x = GlobalAveragePooling2D()(base_model.output)
    return Model(base_model.input, x, name="CNN_Branch")

# LSTM Branch for Handcrafted Features
def build_lstm_branch(feature_size=50):
    input_layer = Input(shape=(feature_size, 1))
    x = LSTM(128, return_sequences=False)(input_layer)
    return Model(input_layer, x, name="LSTM_Branch")

# Define Inputs
image_input_1 = Input(shape=(299, 299, 3))
image_input_2 = Input(shape=(299, 299, 3))
handcrafted_input_1 = Input(shape=(50, 1))
handcrafted_input_2 = Input(shape=(50, 1))

# Build CNN and LSTM branches
cnn_branch = build_cnn_branch()
lstm_branch = build_lstm_branch()

# Extract Features
cnn_features_1 = cnn_branch(image_input_1)
cnn_features_2 = cnn_branch(image_input_2)
lstm_features_1 = lstm_branch(handcrafted_input_1)
lstm_features_2 = lstm_branch(handcrafted_input_2)

# Merge Features
merged_1 = Concatenate()([cnn_features_1, lstm_features_1])
merged_2 = Concatenate()([cnn_features_2, lstm_features_2])

# Compute Euclidean Distance
def euclidean_distance(vectors):
    x, y = vectors
    return K.sqrt(K.sum(K.square(x - y), axis=1, keepdims=True))

distance = Lambda(euclidean_distance)([merged_1, merged_2])

# Siamese Model
siamese_model = Model(inputs=[image_input_1, image_input_2, handcrafted_input_1, handcrafted_input_2], outputs=distance)
siamese_model.compile(optimizer="adam", loss="contrastive_loss")

# Summary
siamese_model.summary()


In [ ]:
import tensorflow.keras.backend as K
from tensorflow.keras.losses import Loss

class ContrastiveLoss(Loss):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def call(self, y_true, y_pred):
        y_true = K.cast(y_true, y_pred.dtype)
        square_pred = K.square(y_pred)
        margin_square = K.square(K.maximum(self.margin - y_pred, 0))
        return K.mean((1 - y_true) * square_pred + y_true * margin_square)

# Compile with contrastive loss
siamese_model.compile(loss=ContrastiveLoss(), optimizer="adam")

In [ ]:
siamese_model.fit(train_generator, validation_data=val_generator, epochs=20)